# Feature engineering
In this step we will introduce features learned from the data exploration step, re-formatting the information we have into a language the model can understand. <br>
This includes: <br>
- splitting wind speed into discrete bins to approach the "S" curve <br>
- splitting wind directions into discrete bins (NE, NW, SW, SE) <br>
- create cycling time feature (hour_of_day, month_of_year)


## Import necessary libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, r2_score, mean_absolute_error
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from scipy.stats import randint, uniform

import sys, os, importlib
sys.path.insert(0, os.path.abspath('..'))
import process_data
process_data.pd = pd
process_data.np = np
process_data.Pipeline = Pipeline
process_data.ColumnTransformer = ColumnTransformer
process_data.StandardScaler = StandardScaler
process_data.OneHotEncoder = OneHotEncoder


## Load data

In [ ]:
path1 = "../Data/Processed/wind_persistance_polar.csv"
path2 = "../Data/Processed/wind_average_polar.csv"
path3 = "../Data/Processed/wind_hourly_polar.csv"

def load_data(path):
    data = pd.read_csv(path)
    return data

wind_persistance_polar = load_data(path1)
wind_average_polar = load_data(path2)
wind_hourly_polar = load_data(path3)

# forecast_temp_A = pd.read_csv("../Data/WeatherForecastInput_A.csv") not used anymore
forecast_temp_B = pd.read_csv("../Data/WeatherForecastInput_B.csv")

wind_persistance_polar.info()

Somehow the timestamp is not datetime type 

## Apply features and split input and output data

In [ ]:
# List of features to use
features_to_use = [
    'wind_speed_100m',
    'wind_speed_bin',
    'wind_dir_bin',
    'hour_sin',
    'hour_cos',
    'month_sin',
    'month_cos'
]

# Create features for persistence data
X_pers, y_pers_train = process_data.create_features(wind_persistance_polar)
X_pers = X_pers[features_to_use].copy()

# Create features for average data
X_avg, y_avg_train = process_data.create_features(wind_average_polar)
X_avg = X_avg[features_to_use].copy()

# Create features for hourly data
X_hour, y_hour_train = process_data.create_features(wind_hourly_polar)
X_hour = X_hour[features_to_use].copy()

print("\nDataFrames are ready for preprocessing.")

X_pers.head()

## Machine learning model evaluation

As the wind speed and power output relationship is non-linear, we will compare the results of a Gradient Boosting model (e.g., XGBoost) with a Random Forest model. These will also be compared to a Linear Regression to further exemplify the non-linearity of the data set. <br>
We will evaluate the models performance using cross-validation and their results will be compared using MAE (Mean Average Error) and R² (Coefficient of Determination). <br>
The highest scoring model will be chosen for further fine tuning.

In [ ]:
# XGBoost with tuned parameters for wind power forecasting
xgb_model = XGBRegressor(
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror',
    verbosity=0,
    n_jobs=-1
)

In [ ]:
# Random Forest regressor
rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

In [ ]:
#  Linear Regression
linear_model = LinearRegression()

First we evaluate the three models on the Persistence Fill dataset.

In [ ]:
# Get preprocessing pipeline
preprocessor = process_data.get_preprocessor()

#Results of all ML models for the Persistance Fill data set
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42)
}

scoring = {
    'r2': make_scorer(r2_score),
    'mae': make_scorer(mean_absolute_error, greater_is_better=False)
}

results = {}
for name, model in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
    # Use the raw DataFrame X_pers so ColumnTransformer can select columns by name
    cv_results = cross_validate(pipeline, X_pers, y_pers_train, cv=5, scoring=scoring, n_jobs=-1)
    results[name] = cv_results

# Summarize results into a small DataFrame and print
summary_list = []
for name, res in results.items():
    mean_r2 = np.mean(res['test_r2'])
    mean_mae = np.mean(res['test_mae'])
    # If MAE was returned as negative (scoring with greater_is_better=False), take absolute
    if mean_mae < 0:
        mean_mae = -mean_mae
    summary_list.append({'model': name, 'mean_r2': mean_r2, 'mean_mae': mean_mae})

print("Results for Persistance Fill")
summary_df = pd.DataFrame(summary_list).set_index('model').sort_values('mean_r2', ascending=False)
print(summary_df)


Now we evaluate the three models on the Moving Average Fill dataset.

In [ ]:
#Results of all ML models for the Moving Average Fill dataset
results = {}
for name, model in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
    # Use the raw DataFrame X_avg so ColumnTransformer can select columns by name
    cv_results = cross_validate(pipeline, X_avg, y_avg_train, cv=5, scoring=scoring, n_jobs=-1)
    results[name] = cv_results

# Summarize results into a small DataFrame and print
summary_list = []
for name, res in results.items():
    mean_r2 = np.mean(res['test_r2'])
    mean_mae = np.mean(res['test_mae'])
    # If MAE was returned as negative (scoring with greater_is_better=False), take absolute
    if mean_mae < 0:
        mean_mae = -mean_mae
    summary_list.append({'model': name, 'mean_r2': mean_r2, 'mean_mae': mean_mae})
print("Results for Moving Average Fill")
summary_df = pd.DataFrame(summary_list).set_index('model').sort_values('mean_r2', ascending=False)
print(summary_df)

Lastly we evaluate the three models on the Climatology Fill dataset.

In [ ]:
#Results of all ML models for the Hourly Climatology Fill data set
results = {}
for name, model in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
    # Use the raw DataFrame X_hour so ColumnTransformer can select columns by name
    cv_results = cross_validate(pipeline, X_hour, y_hour_train, cv=5, scoring=scoring, n_jobs=-1)
    results[name] = cv_results

# Summarize results into a small DataFrame and print
summary_list = []
for name, res in results.items():
    mean_r2 = np.mean(res['test_r2'])
    mean_mae = np.mean(res['test_mae'])
    # If MAE was returned as negative (scoring with greater_is_better=False), take absolute
    if mean_mae < 0:
        mean_mae = -mean_mae
    summary_list.append({'model': name, 'mean_r2': mean_r2, 'mean_mae': mean_mae})
print("Results for Hourly Climatology Fill")
summary_df = pd.DataFrame(summary_list).set_index('model').sort_values('mean_r2', ascending=False)
print(summary_df)

After cross-validation all models had quite high R2 and low MAE scores. However, the XGBoost model performed the best in both metrics, thus this model is chosen for further fine tuning. Additionally, the R2 score was the highest and MAE the lowest for the Persistence Fill dataset, so this method of filling in the missing power output values is chosen as the best option.

Comment: Experiment with the rolling average and further linearisation of the S-curve some more

### Fine Tuning
The best performing model (XGBoost) will be fine tuned using RandomizedSearchCV to find the optimal hyperparameters to maximize R2 score. <br>

In [ ]:
# Fine-tune XGBoost hyperparameters with RandomizedSearchCV (maximize R2)
from sklearn.pipeline import Pipeline as SKPipeline

# Create a fresh XGBoost estimator inside a pipeline so params can be prefixed with 'model__'
xgb_base = XGBRegressor(objective='reg:squarederror', verbosity=0, n_jobs=-1, random_state=42)
pipeline_rs = SKPipeline([('preprocessor', preprocessor), ('model', xgb_base)])

# Parameter distributions (prefixed with 'model__' to target the estimator in the pipeline)
param_dist = {
    'model__n_estimators': randint(100, 1200),
    'model__max_depth': randint(3, 12),
    'model__learning_rate': uniform(0.01, 0.29),
    'model__subsample': uniform(loc=0.5, scale=0.5),
    'model__colsample_bytree': uniform(loc=0.5, scale=0.5),
    'model__gamma': uniform(loc=0, scale=5),
    'model__min_child_weight': randint(1, 10)
}

# TimeSeries CV to avoid leakage (use the same split count used earlier)
tscv_tune = TimeSeriesSplit(n_splits=5)

# Randomized search (optimize for R2). n_iter set modestly to keep runtime reasonable; increase if you have time.
rs = RandomizedSearchCV(
    estimator=pipeline_rs,
    param_distributions=param_dist,
    n_iter=40,
    scoring='r2',
    cv=tscv_tune,
    random_state=42,
    n_jobs=-1,
    verbose=2,
    return_train_score=False
)

# Fit the search on the Persistence Fill dataset
# This step takes quite a lot of time depending on n_iter and dataset size. May need optimization.

rs.fit(X_pers, y_pers_train)

# Save best estimator and results for later reuse
best_xgb = rs.best_estimator_
best_params = rs.best_params_
best_score = rs.best_score_
print('Best R2 (cv):', best_score)
print('Best params:')
for k, v in best_params.items():
    print(f'  {k}: {v}')



Kristopher:
After fine tuning the XGBoost model using RandomizedSearchCV, the best parameters were found to be:
- n_estimators: 1061
- max_depth: 8
- learning_rate: 0.1884
- subsample: 0.6271
- colsample_bytree: 0.7850
- gamma: 0.4859
- min_child_weight: 7 
<br>


The R2 score achieved with these parameters was 0.779, which is a small, but significant improvement from the previously chosen parameters. <br>
The exact parameters can change for different computer hardware.

In [ ]:
# Summarize top 5 results
cv_results = pd.DataFrame(rs.cv_results_)
top = cv_results.sort_values('mean_test_score', ascending=False).head(5)[[
    'mean_test_score', 'std_test_score', 'params'
]]
print(top)



### Preprocessing test data and evaluating fine-tuned model

Before evaluating the fine-tuned model on the test set, the test data must first be preprocessed using the same preprocessing pipeline as the training data. <br>
The fine-tuned model is then used to make predictions on the preprocessed test data. The performance of the model is evaluated using R².

In [ ]:
# Import processed weather data for test set evaluation
path4 = "../Data/Processed/weather_input_polar.csv"
# Importing the solution weather data to evaluate the fine-tuned model on the test set
path5 = "../Data/Solution_A.csv"

def load_data(path):
    data = pd.read_csv(path)
    return data

weather_input_polar = load_data(path4)
solution_A = load_data(path5)

print(solution_A.info())
print(solution_A.head())

In [ ]:
# Ensure TIMESTAMP columns are comparable (convert in copies only)
weather_input_polar['TIMESTAMP'] = pd.to_datetime(weather_input_polar['TIMESTAMP'])

# Check the result
weather_input_polar.head()


In [ ]:
# 1) Create the same engineered features on the weather input DataFrame
X_weather_input, y_weather_input = process_data.create_features(weather_input_polar)
# Keep the same columns used for training (features_to_use defined earlier)
X_weather_input = X_weather_input[features_to_use].copy()

# 2) Fit the preprocessor on the persistence training data (fit once)
preprocessor = process_data.get_preprocessor()
preprocessor.fit(X_pers) 

### Evaluation of fine-tuned model on test set

In [ ]:
#Evaluate the fine tuned XGBoost model on the preprocessed weather input data set using R2 score
weather_input_test = rs.best_estimator_

# Predict on the power data
final_predictions = weather_input_test.predict(X_weather_input)    

''' Only for evaluation of Weather_Forecast_Input_A
final_r2 = r2_score(solution_A['POWER'], final_predictions)
final_mae = mean_absolute_error(solution_A['POWER'], final_predictions)
print("Final R2 on Weather_Forecast_Input_A test set:", final_r2)
print("Final MAE on Weather_Forecast_Input_A test set:", final_mae)
'''

In [ ]:
# Export predictions to CSV for submission
'''
submission_df = pd.DataFrame({
    'TIMESTAMP': forecast_temp_A['TIMESTAMP'],
    'POWER': final_predictions
})
submission_df.to_csv("../Data/Submission/submission_A.csv", index=False)
'''

# Export predictions for model B
submission_df_B = pd.DataFrame({
    'TIMESTAMP': forecast_temp_B['TIMESTAMP'],
    'FORECAST': final_predictions
})
submission_df_B.to_csv("../Data/Submission/Competition_1_Group_B.csv", index=False)


### Comments:
 It's a bit unintuitive that the previously preprocessed weather input data is being preprocessed again here, but the previous check is useful to ensure that the preprocessing pipeline can handle the missing features in the weather input data.

 The final R2 score for the test dataset (Weather_Forecast_Input_A) is 0.8630 and MAE is 0.0943. Personally I think these results are suspiciously good, since the cross-validated R2 score was only 0.779. Please double check that everything is correct here.

Perhaps the results could be improved by some feature engineering, such as increasing the importance of the power forecast in training specifically for February, since that is the month for which the model is being tested.




In [ ]:
solution_A = pd.read_csv("../Data/Solution_A.csv")
solution_A['TIMESTAMP'] = pd.to_datetime(solution_A['TIMESTAMP'])
final_predictions= pd.read_csv("../Data/Submission/submission_A.csv")
final_predictions['TIMESTAMP'] = pd.to_datetime(final_predictions['TIMESTAMP'])

# Plot submission results and soluation_A in one plot
plt.figure(figsize=(12, 6))
plt.plot(solution_A['TIMESTAMP'], solution_A['POWER'], label='Solution A', color='#005293', alpha=0.8)
plt.plot(final_predictions['TIMESTAMP'], final_predictions['POWER'], label='Predictions', color='#E37222', alpha=0.8)
plt.xlabel('Timestamp')
plt.ylabel('Power')
plt.title('Predictions vs Solution A')
plt.legend()

plt.savefig('../Data/Plots/predictions_vs_solution_A.png', dpi=300)
plt.show()

